<a href="https://colab.research.google.com/github/kamil0sek1/kursAI/blob/main/Titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd # tabele danych
import seaborn as sns # źródło danych
import matplotlib.pyplot as plt # wykresy
from scipy import stats


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


import numpy as np

In [ ]:
titanic = sns.load_dataset("titanic")
print(titanic.head(10))

| Kolumna | Znaczenie |
|---|---|
| `survived` | Czy pasażer przeżył: `0` = nie, `1` = tak |
| `pclass` | Klasa biletu: `1` = pierwsza, `2` = druga, `3` = trzecia |
| `sex` | Płeć pasażera: `male` = mężczyzna, `female` = kobieta |
| `age` | Wiek pasażera |
| `sibsp` | Liczba rodzeństwa lub małżonków na statku |
| `parch` | Liczba rodziców lub dzieci na statku |
| `fare` | Cena biletu |
| `embarked` | Port wejścia na statek zapisany skrótem (`S` - Southampton, `C` - Cherbourg, `Q` - Queenstown)|
| `class` | Klasa biletu zapisana słownie: `First`, `Second`, `Third` |
| `who` | Typ osoby: `man`, `woman` lub `child` |
| `adult_male` | Czy pasażer był dorosłym mężczyzną: `True` / `False` |
| `deck` | Pokład/statkowa sekcja, np. `A`, `B`, `C`; często brakuje tej wartości |
| `embark_town` | Miasto/port wejścia na statek |
| `alive` | Czy pasażer przeżył zapisane tekstowo: `yes` / `no` |
| `alone` | Czy pasażer podróżował sam: `True` / `False` |

In [ ]:
from pandas.io.formats.style_render import Subset
print("\n Brakujące wartości w każdej kolumie: ")
print(titanic.isnull().sum())

liczba_wierszy_przed_czyszczeniem = len(titanic)
titanic_clean = titanic.dropna(subset=["age"])
liczba_wierszy_po_czyszczeniu = len(titanic_clean)
print("="*50)
print("\n Liczba wierszy przed czyszczeniem: ", liczba_wierszy_przed_czyszczeniem)
print("Liczba wierszy po czyszczeniu: ", liczba_wierszy_po_czyszczeniu)
print("Liczba wierszy usuniętych: ", liczba_wierszy_przed_czyszczeniem - liczba_wierszy_po_czyszczeniu)
print("="*50)
print(titanic_clean.isnull().sum())

In [ ]:
#sekcja 3: ceny biletów

def categorize_fare(fare, fare_ranges):
    if fare <= fare_ranges[0]:
        return 0
    elif fare <= fare_ranges[1]:
        return 1
    else:
        return 2


min_fare = titanic_clean["fare"].min()
max_fare = titanic_clean["fare"].max()


print("Min fare: ", min_fare)
print("Max fare: ", max_fare)
fare_step = (max_fare - min_fare) / 3
fare_ranges = [
    min_fare + fare_step,
    min_fare + 2 * fare_step,
]

print("Krok fare: ", fare_step)


# Wyświetlenie przedziałów cenowych
print("\nPrzedziały cenowe biletów:")
print(f"Niska: {min_fare:.2f} - {fare_ranges[0]:.2f}")
print(f"Średnia: {fare_ranges[0]:.2f} - {fare_ranges[1]:.2f}")
print(f"Wysoka: {fare_ranges[1]:.2f} - {max_fare:.2f}")


In [ ]:
# * Które kolumny naprawdę mogą pomóc przewidzieć, czy pasażer przeżył?

# Po co wybieramy tylko niektóre cechy?

# Bo nie każda kolumna pomaga modelowi, bo niektóre dane są:
# - przydatne, bo mają związek z przeżyciem,
# - zbędne, bo powtarzają informacje z innych kolumn,
# - problematyczne, bo mają dużo braków,
# - zakazane, bo zawierają odpowiedź.

# [najważnijesze cechy: sex, pcalss, age, fare_category (fare zmienione na nasze kategorie)]

titanic_model = titanic_clean.copy()
titanic_model["sex"] = titanic_model["sex"].map({
    "male" : 1,
    "female" : 0
})
titanic_model["fare_category"] = titanic_model["fare"].apply(
    lambda fare: categorize_fare(fare, fare_ranges)
)
print(titanic_model["fare_category"].value_counts())
selected_features = ["sex", "pclass", "age", "fare_category"]
print(titanic_model[selected_features].head(10))



In [ ]:
x = titanic_model[selected_features]
y = titanic_model["survived"]
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42

)


scaler = StandardScaler()


x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)


# Sprawdzenie rozmiarów zbiorów
print("Rozmiary zbiorów:")
print(f"X_train: {x_train.shape}")
print(f"X_test: {x_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")

# **Jak działa skalowanie?**

| pasażer | wiek `age` | cena biletu `fare` |
| ------- | ---------: | -----------------: |
| A       |         10 |                  0 |
| B       |         20 |                100 |
| C       |         30 |                500 |


Cena biletu ma dużo większe liczby niż wiek, więc model mógłby za bardzo zwracać uwagę na fare.


| pasażer | `age` po skalowaniu | `fare` po skalowaniu |
| ------- | ------------------: | -------------------: |
| A       |               -1.22 |                -0.93 |
| B       |                0.00 |                -0.46 |
| C       |                1.22 |                 1.39 |


```
jak dane wypadają względenm swojej kolumny
```

In [ ]:
# Sekcja 6: Trenowanie i ocena modelu
#
# Utworzenie modelu regresji logistycznej
# https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html?utm_source=chatgpt.com

# z = w1 * sex + w2 * pclass + w3 * age + w4 * fare_category + b

# wagi określają: co się dzieje z szansą przeżycia, gdy wartość tej cechy rośnie

# p = 1 / (1 + e^(-z))


model = LogisticRegression(
    random_state = 42,
    C = 0.2
)

model.fit(x_train_scaled, y_train)
print("Liczba wykonanych iteracji:", model.n_iter_)

# Wyświetlenie wag modelu
wagi_modelu = pd.DataFrame({
    "cecha": selected_features,
    "waga": model.coef_[0]
})

print("\nWagi, których nauczył się model:")
print(wagi_modelu)

# Wyświetlenie wyrazu wolnego
print("\nWyraz wolny modelu:")
print(model.intercept_[0])


y_pred = model.predict(x_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
print("\nWynik modelu:")
print(f"Dokładnosć modelu: {accuracy:.3f}")

print("\nRaport klasyfikacji:")
print(classification_report(y_test, y_pred))

In [ ]:
conf_matrix = confusion_matrix(
    y_test,
    y_pred,
    labels=[1,0]
)

# Zamiana macierzy pomyłek na czytelną tabelę
conf_matrix_df = pd.DataFrame(
    conf_matrix,
    index=["Prawda: przeżył (1)", "Prawda: nie przeżył (0)"],
    columns=["Model: przeżył (1)", "Model: nie przeżył (0)"]
)

print("\nMacierz pomyłek:")
print(conf_matrix_df)

print("="*70)

# Wizualizacja macierzy pomyłek

plt.figure(figsize=(7, 5))

sns.heatmap(
    conf_matrix_df,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("Macierz pomyłek - Titanic")
plt.xlabel("Przewidywanie modelu")
plt.ylabel("Prawdziwa wartość")
plt.show()

In [ ]:
# Sekcja 7: Interaktywny symulator

while True:
  print("="* 70)
  print("=== Prognoza przeżycia pasażera ===")
  print("Wpisz 'koniec', aby zakończyć program.")

  try:
    sex_input = input("\nPodaj płeć (m/k): ").lower()
    if sex_input == "koniec":
      break
    sex = 0 if sex_input == 'k' else 1


    pclass_input = input("Podaj klasę podóży (1/2/3):")
    if pclass_input == "koniec":
      break
    pclass = int(pclass_input)


    age_input = input("Podaj wiek: ")
    if age_input == "koniec":
      break
    age = int(age_input)


    print("\nKategorie cenowe biletów:")
    print(f"0 - Niska:   {min_fare:.2f} - {fare_ranges[0]:.2f}")
    print(f"1 - Średnia: {fare_ranges[0]:.2f} - {fare_ranges[1]:.2f}")
    print(f"2 - Wysoka:  {fare_ranges[1]:.2f} - {max_fare:.2f}")

    fare_category_input = input("Podaj kategorię biletu (0/1/2): ")
    if fare_category_input == "koniec":
      break
    fare_category = int(fare_category_input)

    new_passager = pd.DataFrame(
        [[sex, pclass, age, fare_category]],
        columns=selected_features
    )


    new_passager_scalled = scaler.transform(new_passager)
    prediction = model.predict(new_passager_scalled)
    probability = model.predict_proba(new_passager_scalled)

    print("\nWyniki prognozy:")
    print(f"Płeć: {'Kobieta' if sex == 0 else 'Mężczyzna'}")
    print(f"Klasa podróży: {pclass}")
    print(f"Wiek: {age}")
    print(f"Kategoria cenowa: {fare_category}")
    print(f"Przewidywanie przeżycia: {'Tak' if prediction[0] == 1 else 'Nie'}")
    print(f"Prawdopodobieństwo przeżycia: {probability[0][1]:.2%}")

  except ValueError:
    print(f"\n wpisz poprawną liczbę")


  dalej = input("\n Czy chcesz sprawdzić kolejnego pasażera? (t/n)").lower()
  if dalej != "t":
    break



print("\nKoniec programu")